# Lab 5: Gaussian Blur
Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Run all**.
Có thể tải `image.jpg` vào Files trước khi chạy; nếu không có, notebook dùng ảnh mẫu có sẵn.

Kết quả: ảnh, `metrics.csv`, báo cáo LaTeX và file `outputs/Lab5.zip`.
Thời gian GPU đã warm-up, đồng bộ; không tính truyền dữ liệu. Dùng GPU thật để đo speedup.


In [ ]:
%pip -q install "numba-cuda[cu12]"


In [ ]:
import time
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from numba import cuda, float32, float64

OUT = Path('outputs/Lab5')
OUT.mkdir(parents=True, exist_ok=True)

assert cuda.is_available(), 'Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.'
device = cuda.get_current_device()
print('GPU:', device.name)
MAX_THREADS = device.MAX_THREADS_PER_BLOCK
BLOCKS = [b for b in [(8, 8), (16, 16), (32, 16), (32, 32)]
          if b[0] * b[1] <= MAX_THREADS]
REPEATS = 3

def load_image(path='image.jpg'):
    if Path(path).exists():
        a = plt.imread(path)
        if np.issubdtype(a.dtype, np.floating):
            a = np.clip(a * 255, 0, 255).astype(np.uint8)
        return np.asarray(Image.fromarray(a).convert('RGB')).copy()
    # Ảnh mẫu có màu, cạnh và nhiễu; không cần tải file để Run all.
    y, x = np.indices((256, 256))
    a = np.stack((x, y, (x // 32 % 2) * 160 + 40), axis=-1)
    noise = np.random.default_rng(0).integers(-15, 16, a.shape)
    return np.clip(a + noise, 0, 255).astype(np.uint8)

image = load_image()
H, W = image.shape[:2]
gray = (0.299 * image[:, :, 0] + 0.587 * image[:, :, 1]
        + 0.114 * image[:, :, 2]).astype(np.uint8)
print('Ảnh:', image.shape)

def grid(block):
    return ((W + block[0] - 1) // block[0],
            (H + block[1] - 1) // block[1])

def gpu_time(run):
    run()  # Warm-up: không tính biên dịch JIT vào thời gian.
    cuda.synchronize()
    start = time.time()
    for _ in range(REPEATS):
        run()
    cuda.synchronize()
    return (time.time() - start) / REPEATS

def cpu_time(run):
    start = time.time()
    result = run()
    return result, time.time() - start

def show_images(items):
    fig, axes = plt.subplots(1, len(items), figsize=(4 * len(items), 4), squeeze=False)
    for ax, (title, a) in zip(axes[0], items):
        ax.imshow(a, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(OUT / 'images.png', dpi=120)
    plt.show()

def save_image(name, a):
    Image.fromarray(a).save(OUT / name)

def save_results(rows, report_name, report_text):
    df = pd.DataFrame(rows)
    df.to_csv(OUT / 'metrics.csv', index=False)
    print(df.to_string(index=False))
    # Bảng LaTeX nhỏ, không cần thêm thư viện dựng báo cáo.
    def tex(v):
        return str(v).replace('_', r'\_')
    lines = [r'\begin{center}', r'\resizebox{\linewidth}{!}{',
             r'\begin{tabular}{' + 'l' * len(df.columns) + '}', r'\hline',
             ' & '.join(tex(c) for c in df.columns) + r' \\', r'\hline']
    for row in df.itertuples(index=False, name=None):
        lines.append(' & '.join(f'{v:.6g}' if isinstance(v, float) else tex(v)
                                for v in row) + r' \\')
    lines += [r'\hline', r'\end{tabular}', '}', r'\end{center}']
    (OUT / 'results.tex').write_text('\n'.join(lines), encoding='utf-8')
    (OUT / report_name).write_text(report_text, encoding='utf-8')
    shutil.make_archive(str(OUT), 'zip', OUT)
    print('Kết quả và báo cáo:', str(OUT) + '.zip (tải trong Files bên trái)')


In [ ]:
FILTER = np.array([
    [0, 0, 1, 2, 1, 0, 0], [0, 3, 13, 22, 13, 3, 0],
    [1, 13, 59, 97, 59, 13, 1], [2, 22, 97, 159, 97, 22, 2],
    [1, 13, 59, 97, 59, 13, 1], [0, 3, 13, 22, 13, 3, 0],
    [0, 0, 1, 2, 1, 0, 0]], dtype=np.float32)
assert FILTER.sum() == 1003

def blur_cpu(src):
    out = src.copy()
    for y in range(3, src.shape[0] - 3):
        for x in range(3, src.shape[1] - 3):
            out[y, x] = int(np.sum(src[y-3:y+4, x-3:x+4] * FILTER) / 1003.0)
    return out

@cuda.jit
def blur_global(src, dst, weights):
    x, y = cuda.grid(2)
    h, w = src.shape
    if x < w and y < h:
        if 3 <= x < w - 3 and 3 <= y < h - 3:
            total = 0.0
            for j in range(7):
                for i in range(7):
                    total += src[y + j - 3, x + i - 3] * weights[j, i]
            dst[y, x] = int(total / 1003.0)
        else:
            dst[y, x] = src[y, x]

@cuda.jit
def blur_shared(src, dst, weights):
    filt = cuda.shared.array((7, 7), float32)
    tid = cuda.threadIdx.y * cuda.blockDim.x + cuda.threadIdx.x
    count = cuda.blockDim.x * cuda.blockDim.y
    for i in range(tid, 49, count):
        filt[i // 7, i % 7] = weights[i // 7, i % 7]
    cuda.syncthreads()  # Mọi thread, kể cả thread ngoài ảnh, đều phải đến barrier.
    x, y = cuda.grid(2)
    h, w = src.shape
    if x < w and y < h:
        if 3 <= x < w - 3 and 3 <= y < h - 3:
            total = 0.0
            for j in range(7):
                for i in range(7):
                    total += src[y + j - 3, x + i - 3] * filt[j, i]
            dst[y, x] = int(total / 1003.0)
        else:
            dst[y, x] = src[y, x]


In [ ]:
reference, cpu_s = cpu_time(lambda: blur_cpu(gray))
d_src, d_filter = cuda.to_device(gray), cuda.to_device(FILTER)
d_dst = cuda.device_array_like(d_src)
rows = []
outputs = {}
for block in BLOCKS:
    for name, kernel in [('global', blur_global), ('shared', blur_shared)]:
        seconds = gpu_time(lambda: kernel[grid(block), block](d_src, d_dst, d_filter))
        result = d_dst.copy_to_host()
        error = int(np.abs(result.astype(int) - reference.astype(int)).max())
        assert error <= 1
        outputs[name] = result
        rows.append(dict(block=str(block), mode=name, cpu_ms=cpu_s * 1000,
                         gpu_ms=seconds * 1000, speedup=cpu_s / seconds, max_error=error))
for name, result in outputs.items():
    save_image('image_blur_' + name + '.png', result)
save_image('image_blur_cpu.png', reference)
show_images([('Input', gray), ('Global', outputs['global']), ('Shared filter', outputs['shared'])])
plt.figure(figsize=(6, 3))
for name in ['global', 'shared']:
    data = [r for r in rows if r['mode'] == name]
    plt.plot([r['block'] for r in data], [r['speedup'] for r in data], 'o-', label=name)
plt.xlabel('2D block'); plt.ylabel('CPU / GPU speedup'); plt.legend(); plt.grid()
plt.tight_layout(); plt.savefig(OUT / 'performance.png', dpi=120); plt.show()


In [ ]:
report_text = r"""
\documentclass{article}
\usepackage[utf8]{inputenc}
\usepackage[T5]{fontenc}
\usepackage{graphicx}
\usepackage[margin=1.5cm]{geometry}
\title{Lab 5: Gaussian Blur}
\author{Nguyễn Thanh Hùng -- 2540056}
\date{}
\begin{document}
\maketitle
A $7\times7$ Gaussian filter with coefficient sum 1003 is applied to grayscale pixels.
The global version reads the filter from device memory; the shared version cooperatively loads its 49 coefficients and synchronizes before convolution.
The three-pixel border is preserved. A CPU reference validates both versions.
The graph compares speedup for different 2D blocks. Shared memory has loading and synchronization costs, so its benefit must be measured.

Kernel timing uses \texttt{time.time()} after JIT warm-up and CUDA synchronization,
averaged over three GPU runs. Device allocation and host/device copies are excluded.
CPU measurements are single runs. Speedup is relative to the stated CPU implementation.
Actual measurements appear after running the notebook on Colab; no timings are assumed here.
\IfFileExists{results.tex}{\input{results.tex}}{}
\IfFileExists{performance.png}{\begin{center}\includegraphics[width=.75\linewidth]{performance.png}\end{center}}{}
\IfFileExists{images.png}{\begin{center}\includegraphics[width=\linewidth]{images.png}\end{center}}{}
\IfFileExists{histograms.png}{\begin{center}\includegraphics[width=.8\linewidth]{histograms.png}\end{center}}{}
\end{document}
"""
save_results(rows, 'Report.5.gaussian.blur.tex', report_text)
